# TrainingWeightedTokens

Two named loss configurations, selected in the configuration cell:
- 1stWeightConfig: decisions 1.0, structure/EOS 0.1, fixed field names 0.0.
- 2ndWeightConfig (selected): the whole assistant JSON and EOS have weight 1.0.

User/image/header tokens remain masked with -100 and weight 0.0 in both cases. Keep the complete assistant JSON in the input sequence. Only loss weights change; model, LoRA, sample and 100 epochs stay the same.

Use AppliedTaskByte (.venvAppliedTask). Restart the kernel before running the second experiment so it starts from the same base model, not the first run's adapters. The updated notebook has not been executed by Codex. The previous executed notebook and outputs are preserved in TrainingWeightedTokens_1stWeightConfig.ipynb; existing saved runs/adapters are untouched.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "models.json").is_file():
    raise FileNotFoundError("Start in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
import importlib
import weighted_tokens_helpers
importlib.reload(weighted_tokens_helpers)
from weighted_tokens_helpers import load_joint_sample, build_masked_target, collate_weighted_tokens, weighted_supervised_loss

SAMPLE_PATHS = [ROOT / "data/training/joint_targets/seq_71033fec4de494fc_page_01/sample.json"]
samples = [load_joint_sample(path) for path in SAMPLE_PATHS]
print("Interpreter:", sys.executable)
print("Samples:", [sample["page_id"] for sample in samples])


Interpreter: c:\Users\ESHAAN\BYTE\Applied-Task\.venvAppliedTask\Scripts\python.exe
Samples: ['seq_71033fec4de494fc_page_01']


## Loss and LoRA configuration

Change WEIGHT_CONFIG_NAME to select either named configuration. Each uses its own RUN_NAMES entry. Existing run directories cannot be overwritten. For another repetition, choose a new run name in RUN_NAMES.

2ndWeightConfig gives decision IDs, JSON punctuation, fixed field names and EOS weight 1.0. This is ordinary assistant-only cross-entropy through our existing weighted-loss function. 1stWeightConfig preserves the earlier values exactly.

Keep q/k/v/o LoRA in the final eight LLM layers, rank 4, alpha 8, dropout 0.05, one image tile, learning rate 1e-4 and 100 epochs. Vision, projector and base LLM weights remain frozen. The single page is a smoke experiment, not evidence of generalization.


In [8]:
# Select one named loss configuration. The old weights remain available.
WEIGHT_CONFIGS = {
    "1stWeightConfig": {
        "decision_weight": 1.0,
        "formatting_weight": 0.1,
        "end_weight": 0.1,
        "field_weight": 0.0,
        "ignore_index": -100,
        "train_decisions": ["P IDs", "A IDs", "B IDs", "UNKNOWN"],
    },
    "2ndWeightConfig": {
        "decision_weight": 1.0,
        "formatting_weight": 1.0,
        "end_weight": 1.0,
        "field_weight": 1.0,
        "ignore_index": -100,
        "train_decisions": ["P IDs", "A IDs", "B IDs", "UNKNOWN"],
    },
}
RUN_NAMES = {
    "1stWeightConfig": "weighted_tokens_structure_smoke_01",
    "2ndWeightConfig": "weighted_tokens_full_assistant_smoke_02",
}
WEIGHT_CONFIG_NAME = "2ndWeightConfig"
LOSS_CONFIG = WEIGHT_CONFIGS[WEIGHT_CONFIG_NAME].copy()

# Model, LoRA, data and training settings match the previous 100-epoch run.
CONFIG = H.experiment(
    name=RUN_NAMES[WEIGHT_CONFIG_NAME],
    decoder="attention",  # q_proj, k_proj, v_proj, o_proj
    last_layers=8,
    projector=False,
    vision=False,
    rank=4,
    epochs=100,
    lr=1e-4,
    max_tiles=1,
    accumulation=1,
)
CONFIG["view"] = "annotated"
CONFIG["max_length"] = 4096
CONFIG["loss"] = LOSS_CONFIG
CONFIG["weight_config_name"] = WEIGHT_CONFIG_NAME
RUN = ROOT / "data/training/runs" / CONFIG["name"]
print("Selected weights:", WEIGHT_CONFIG_NAME)
print(json.dumps(CONFIG, indent=2))


Selected weights: 2ndWeightConfig
{
  "name": "weighted_tokens_full_assistant_smoke_02",
  "decoder": "attention",
  "last_layers": 8,
  "projector": false,
  "vision": false,
  "vision_last_layers": 4,
  "rank": 4,
  "alpha": 8,
  "dropout": 0.05,
  "epochs": 100,
  "lr": 0.0001,
  "max_tiles": 1,
  "view": "annotated",
  "max_length": 4096,
  "accumulation": 1,
  "seed": 42,
  "loss": {
    "decision_weight": 1.0,
    "formatting_weight": 1.0,
    "end_weight": 1.0,
    "field_weight": 1.0,
    "ignore_index": -100,
    "train_decisions": [
      "P IDs",
      "A IDs",
      "B IDs",
      "UNKNOWN"
    ]
  },
  "weight_config_name": "2ndWeightConfig"
}


## Inspect the mask before loading model weights

The preview uses the selected LOSS_CONFIG. With 2ndWeightConfig, every assistant JSON token and the closing chat separator should show weight 1.0. With 1stWeightConfig, field-name tokens show 0.0 and punctuation/EOS show 0.1.

Tokens overlapping an ID receive decision_weight; tokens overlapping a fixed key receive field_weight; remaining JSON tokens receive formatting_weight. Mixed token pieces are weighted as whole tokens. Fast/model tokenizer ID agreement is still checked by the collator.


In [10]:
from transformers import AutoTokenizer

offset_tokenizer = AutoTokenizer.from_pretrained(
    str(H.MODEL_DIR), trust_remote_code=True,
    local_files_only=True, use_fast=True,
)
preview = build_masked_target(offset_tokenizer, samples[0]["target"], separator="<|im_end|>", loss_config=LOSS_CONFIG)
print("Assistant JSON:", preview["text"])
print("Decision spans:", [preview["text"][start:end] for start, end in preview["spans"]])
print("\nToken                     Weight")
for weight, (start, end) in zip(preview["token_weights"], preview["offsets"]):
    piece = (preview["text"] + "<|im_end|>")[start:end]
    print(f"{piece!r:25} {weight:.2f}")


Assistant JSON: {"characters":{"P3":["A4","A5","A7"],"P2":["A6","A2"],"P1":["A1","A3"]},"rejected_appearances":[],"bubble_speakers":{"B10":"P3","B7":"P3","B8":"P3","B6":"P2","B2":"P1","B13":"P1","B9":"P1","B1":"P1","B5":"P1","B12":"P1","B4":"P1","B14":"P1","B3":"P3","B11":"P1"}}
Decision spans: ['P3', 'A4', 'A5', 'A7', 'P2', 'A6', 'A2', 'P1', 'A1', 'A3', 'B10', 'P3', 'B7', 'P3', 'B8', 'P3', 'B6', 'P2', 'B2', 'P1', 'B13', 'P1', 'B9', 'P1', 'B1', 'P1', 'B5', 'P1', 'B12', 'P1', 'B4', 'P1', 'B14', 'P1', 'B3', 'P3', 'B11', 'P1']

Token                     Weight
'{"'                      1.00
'characters'              1.00
'":{"'                    1.00
'P'                       1.00
'3'                       1.00
'":["'                    1.00
'A'                       1.00
'4'                       1.00
'","'                     1.00
'A'                       1.00
'5'                       1.00
'","'                     1.00
'A'                       1.00
'7'                       1.00
'"

## Load the local model and attach LoRA

Run this cell only when ready to use the GPU. The base model remains C:/MODELS/InternVL3-2B.


In [ ]:
import random
import torch

if RUN.exists():
    raise FileExistsError(f"Choose a new run name; {RUN} already exists.")
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
H.cleanup()
model, tokenizer = H.load_base()
parts = H.attach_lora(model, CONFIG)
print(parts)
assert not any(p.requires_grad for p in model.vision_model.parameters())
assert not any(p.requires_grad for p in model.mlp1.parameters())

def collator(items):
    return collate_weighted_tokens(items, model, tokenizer, offset_tokenizer, CONFIG)

batch = collator([samples[0]])
assert (batch["labels"][:, :batch["answer_start"]] == -100).all()
print("Total tokens:", batch["input_ids"].numel())
assert (batch["token_weights"][:, :batch["answer_start"]] == 0).all()
print("Total loss weight:", float(batch["token_weights"].sum()))
print("All supervised tokens:", int((batch["token_weights"] > 0).sum()))
del batch


Loading local base model directly on CUDA...
[{'part': 'language_model', 'trainable': 311296}, {'part': 'mlp1', 'trainable': 0}, {'part': 'vision_model', 'trainable': 0}]
Total tokens: 923
Total loss weight: 136.0
All supervised tokens: 136


## Train the one-page smoke experiment

Use weighted_supervised_loss: unreduced causal cross-entropy multiplied by the collator's per-position weights and normalized by their sum. Labels at position t are predicted from hidden state t-1; weights stay aligned with those labels. The model sees the full JSON; ignored tokens still provide teacher-forced context.

The DataLoader uses our new collator and batch size one. Accumulation supports a larger effective batch. Only LoRA parameters enter the optimizer. This loop does not claim unseen-page accuracy.


In [12]:
from torch.utils.data import DataLoader

if LOSS_CONFIG["ignore_index"] != -100:
    raise ValueError("Our label mask uses ignore_index=-100.")
if RUN.exists():
    raise FileExistsError("Choose a new run name before training.")
RUN.mkdir(parents=True)
H.write_json(RUN / "config.json", CONFIG)
H.write_json(RUN / "sample_paths.json", [str(path) for path in SAMPLE_PATHS])
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=CONFIG["lr"])
loader = DataLoader(samples, batch_size=1, shuffle=True, collate_fn=collator, num_workers=0)
history = []
first_update = True
accumulation = CONFIG["accumulation"]
H.training_mode(model)

for epoch in range(CONFIG["epochs"]):
    optimizer.zero_grad(set_to_none=True)
    epoch_loss = 0.0
    for step, batch in enumerate(loader):
        loss = weighted_supervised_loss(model, batch)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss. Stop and inspect the sample/model.")
        epoch_loss += float(loss.detach())
        # Normalize a final incomplete accumulation group correctly.
        group_start = (step // accumulation) * accumulation
        group_size = min(accumulation, len(loader) - group_start)
        (loss / group_size).backward()
        if (step + 1) % accumulation == 0 or step + 1 == len(loader):
            if first_update:
                report = H.grad_report(model)
                print("First-update gradients:", report)
                if report.get("language_model", 0.0) <= 0:
                    raise RuntimeError("No language LoRA gradient.")
                first_update = False
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        del batch, loss
    row = {"epoch": epoch + 1, "weighted_token_loss": epoch_loss / len(loader)}
    history.append(row)
    print(row)
    H.write_json(RUN / "loss_history.json", history)




First-update gradients: {'language_model': 1.9337567454203963}
{'epoch': 1, 'weighted_token_loss': 0.8914653658866882}
{'epoch': 2, 'weighted_token_loss': 0.8883926272392273}
{'epoch': 3, 'weighted_token_loss': 0.8878359794616699}
{'epoch': 4, 'weighted_token_loss': 0.885467529296875}
{'epoch': 5, 'weighted_token_loss': 0.8721466064453125}
{'epoch': 6, 'weighted_token_loss': 0.866653561592102}
{'epoch': 7, 'weighted_token_loss': 0.8563610315322876}
{'epoch': 8, 'weighted_token_loss': 0.8421738743782043}
{'epoch': 9, 'weighted_token_loss': 0.8286784887313843}
{'epoch': 10, 'weighted_token_loss': 0.8186600208282471}
{'epoch': 11, 'weighted_token_loss': 0.8046663403511047}
{'epoch': 12, 'weighted_token_loss': 0.7919715642929077}
{'epoch': 13, 'weighted_token_loss': 0.7847142815589905}
{'epoch': 14, 'weighted_token_loss': 0.7661257386207581}
{'epoch': 15, 'weighted_token_loss': 0.7583497762680054}
{'epoch': 16, 'weighted_token_loss': 0.7416543364524841}
{'epoch': 17, 'weighted_token_loss':

In [13]:
print(model.config._name_or_path)
print(model.config.model_type)
print(model.__class__.__name__)

C:\MODELS\InternVL3-2B
internvl_chat
InternVLChatModel


## Generate and inspect the result

Compare JSON validity, exact smoke target and termination between the two saved runs. Do not compare their loss values directly: changing weights changes the objective. 2ndWeightConfig tests whether full assistant supervision can reproduce the exact training sample and end the response.

P IDs are arbitrary, so exact target equality is only an overfit diagnostic. More pages and sequence-separated validation are needed to assess generalization.


In [14]:
model.eval()
sample = samples[0]
pixels = H.tiles(sample, CONFIG["max_tiles"], view="annotated")
pixels = pixels.to(device="cuda", dtype=next(model.vision_model.parameters()).dtype)
with torch.inference_mode():
    response = model.chat(tokenizer, pixels, sample["prompt"],
                          {"max_new_tokens": 1024, "do_sample": False})
print(response)
try:
    prediction = json.loads(response)
    print("Valid JSON:", True)
    print("Exact smoke target:", prediction == sample["target"])
except json.JSONDecodeError:
    prediction = None
    print("Valid JSON:", False)
H.write_json(RUN / "smoke_prediction.json", {"response": response, "prediction": prediction})
del pixels


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{"characters":{"P3":["A4","A5","A7"],"P2":["A6","A2"],"P1":["A1","A3"]},"rejected_appearances":[],"bubble_speakers":{"B10":"P3","B7":"P3","B8":"P3","B6":"P2","B2":"P1","B13":"P1","B9":"P1","B1":"P1","B5":"P1","B12":"P1","B4":"P1","B14":"P1","B3":"P3","B11":"P1"}}
Valid JSON: True
Exact smoke target: True


In [ ]:
# H.save_adapters(model, RUN, CONFIG)
# print("Saved adapters:", RUN)

References: [tokenizer offset mappings](https://huggingface.co/docs/transformers/main_classes/tokenizer), [PyTorch cross-entropy](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.cross_entropy.html), [PEFT LoRA configuration](https://huggingface.co/docs/peft/package_reference/lora).
